# Google Drive 세팅 가이드

**train.ipynb 실행 전에 이 노트북을 먼저 실행하세요.**

완료 후 Drive 구조:
```
MyDrive/
└── allergydata/
    ├── data/
    │   ├── processed/          ← 전처리된 이미지 업로드 위치
    │   │   ├── 김치찌개/
    │   │   ├── 된장찌개/
    │   │   └── ...(150 클래스)
    │   └── label_ingredient_map.json
    ├── checkpoints/            ← 학습 중 자동 저장
    └── models/                 ← 최종 모델 저장
        ├── model_fp32.onnx
        ├── model_int8.onnx
        ├── faiss_index.bin
        └── faiss_labels.json
```

## Step 1. Drive 마운트 + 폴더 생성

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DIRS = [
    '/content/drive/MyDrive/allergydata/data/processed',
    '/content/drive/MyDrive/allergydata/checkpoints',
    '/content/drive/MyDrive/allergydata/models',
]

for d in DIRS:
    os.makedirs(d, exist_ok=True)
    print(f'✅ {d}')

print('\n폴더 생성 완료!')

## Step 2. 로컬 → Drive 업로드

아래 두 가지 방법 중 하나를 선택하세요.

---

### 방법 A — Google Drive 웹사이트에서 직접 업로드 (권장)

1. [drive.google.com](https://drive.google.com) 접속
2. `allergydata/data/processed/` 폴더로 이동
3. 로컬 `D:/noallergyforeveryone/data/processed/` 폴더를 통째로 드래그 앤 드롭
4. 업로드 완료 후 아래 셀 실행하여 확인

---

### 방법 B — Colab 파일 업로드 (소량일 때)

```python
from google.colab import files
uploaded = files.upload()  # 파일 선택 다이얼로그
```

## Step 3. 업로드 확인

In [ ]:
DATA_DIR = '/content/drive/MyDrive/allergydata/data/processed'

classes = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
])

print(f'클래스 수: {len(classes)}')
print()

total = 0
short = []  # 이미지 50장 미만 클래스

for cls in classes:
    imgs = os.listdir(os.path.join(DATA_DIR, cls))
    count = len(imgs)
    total += count
    if count < 50:
        short.append((cls, count))

print(f'총 이미지: {total:,}장')
print(f'클래스당 평균: {total // max(len(classes), 1)}장')

if short:
    print(f'\n⚠️  이미지 부족 클래스 ({len(short)}개):')
    for cls, cnt in short:
        print(f'  {cls}: {cnt}장')
else:
    print('\n✅ 모든 클래스 50장 이상 확인')

## Step 4. label_ingredient_map.json 업로드 확인

In [ ]:
import json

MAP_PATH = '/content/drive/MyDrive/allergydata/data/label_ingredient_map.json'

if not os.path.exists(MAP_PATH):
    print('❌ label_ingredient_map.json 없음 → 로컬에서 업로드 필요')
    print('   경로: D:/noallergyforeveryone/data/label_ingredient_map.json')
else:
    with open(MAP_PATH, encoding='utf-8') as f:
        ing_map = json.load(f)

    # 이미지 클래스 중 매핑 없는 것 확인
    missing = [cls for cls in classes if cls not in ing_map]

    print(f'✅ label_ingredient_map.json 로드 완료 ({len(ing_map)}개 클래스)')
    if missing:
        print(f'\n⚠️  알레르기 매핑 없는 클래스 ({len(missing)}개):')
        for m in missing:
            print(f'  {m}')
    else:
        print('✅ 모든 클래스 알레르기 매핑 완료')

## Step 5. GPU 확인

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {gpu}  VRAM: {vram:.1f}GB')

    if 'T4' in gpu:
        print('✅ T4 확인 — FP16 + Batch 64 설정 그대로 사용')
    elif 'A100' in gpu or 'V100' in gpu:
        print('🚀 고성능 GPU — Batch 128로 늘려도 됨')
    else:
        print(f'⚠️  {gpu} — VRAM에 따라 배치 크기 조정 필요할 수 있음')
else:
    print('❌ GPU 없음 — 런타임 → 런타임 유형 변경 → T4 GPU 선택')

## Step 6. 세션 보호 설정 (자동 클릭 방지)

Colab 무료는 일정 시간 미조작 시 세션이 끊깁니다.  
브라우저 콘솔(`F12`)에서 아래 코드를 붙여넣으면 자동으로 연결을 유지합니다.

```javascript
// 브라우저 콘솔에서 실행 (F12 → Console)
function keepAlive() {
  document.querySelector('#top-toolbar colab-connect-button')
    ?.shadowRoot?.querySelector('#connect')?.click();
}
setInterval(keepAlive, 60000);  // 1분마다 실행
```

---

## ✅ 체크리스트 완료 시 train.ipynb 실행

- [ ] Drive 폴더 구조 생성
- [ ] `processed/` 이미지 업로드 (150 클래스)
- [ ] `label_ingredient_map.json` 업로드
- [ ] GPU T4 확인
- [ ] 세션 보호 설정